<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session8_Automated_verification_of_649_zeros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install python-flint==0.9.0

In [2]:
from flint import (
    acb,
    acb_series,
    arb,
    arb_series,
    ctx
)

from fractions import Fraction
import csv
import json
import time

ctx.prec = 192

T_LO = Fraction(10)
T_HI = Fraction(1000)

print("SESSION 08 INITIALIZED")
print("FLINT version: 0.9.0")
print("Precision:", ctx.prec, "bits")
print("Height range: 10 to 1000")

SESSION 08 INITIALIZED
FLINT version: 0.9.0
Precision: 192 bits
Height range: 10 to 1000


In [3]:
#Rigorous arthmetic and contour
def fraction_ball(q):
    return arb(f"{q.numerator}/{q.denominator}")


def interval_ball(a, b):
    midpoint = (a + b) / 2
    radius = abs(b - a) / 2

    return arb(
        f"{midpoint.numerator}/{midpoint.denominator}",
        f"{radius.numerator}/{radius.denominator}"
    )


def segment_box(p, q):
    return acb(
        interval_ball(p[0], q[0]),
        interval_ball(p[1], q[1])
    )


corners = [
    (Fraction(0), T_LO),
    (Fraction(1), T_LO),
    (Fraction(1), T_HI),
    (Fraction(0), T_HI),
    (Fraction(0), T_LO)
]

print("Contour constructed counterclockwise.")
print("Left boundary: Re(s) = 0")
print("Right boundary: Re(s) = 1")
print("Bottom: Im(s) = 10")
print("Top: Im(s) = 1000")

Contour constructed counterclockwise.
Left boundary: Re(s) = 0
Right boundary: Re(s) = 1
Bottom: Im(s) = 10
Top: Im(s) = 1000


In [4]:
#Certify the complete boundary
MAX_DEPTH = 28

stats = {
    "certified_pieces": 0,
    "max_depth": 0,
    "unresolved": []
}


def certify_segment(p, q, depth=0):

    stats["max_depth"] = max(
        stats["max_depth"], depth
    )

    zeta_box = segment_box(p, q).zeta()
    lower = zeta_box.abs_lower()

    if lower.is_finite() and not lower.contains(0):
        stats["certified_pieces"] += 1
        return True

    if depth >= MAX_DEPTH:
        stats["unresolved"].append((p, q))
        return False

    midpoint = (
        (p[0] + q[0]) / 2,
        (p[1] + q[1]) / 2
    )

    left_ok = certify_segment(p, midpoint, depth + 1)
    right_ok = certify_segment(midpoint, q, depth + 1)

    return left_ok and right_ok


boundary_certified = True

print("SESSION 08: BOUNDARY CERTIFICATION")
print("----------------------------------")

for i in range(4):

    result = certify_segment(
        corners[i],
        corners[i + 1]
    )

    print(
        f"Side {i + 1}:",
        "CERTIFIED" if result else "NOT CERTIFIED"
    )

    boundary_certified = boundary_certified and result


print("\nBoundary zero-free:", boundary_certified)
print("Certified pieces:", stats["certified_pieces"])
print("Maximum depth:", stats["max_depth"])
print("Unresolved:", len(stats["unresolved"]))

if not boundary_certified:
    raise RuntimeError(
        "Boundary certification failed. Stop here."
    )


SESSION 08: BOUNDARY CERTIFICATION
----------------------------------
Side 1: CERTIFIED
Side 2: CERTIFIED
Side 3: CERTIFIED
Side 4: CERTIFIED

Boundary zero-free: True
Certified pieces: 51167
Maximum depth: 18
Unresolved: 0


In [5]:
#Automated rigorous contour integration
def zeta_log_derivative(z, analytic):
    jet = acb_series([z, 1], prec=2).zeta()
    coeff = jet.coeffs()

    return coeff[1] / coeff[0]


def interpolate_point(p, q, k, n):
    fraction = Fraction(k, n)

    return (
        p[0] + (q[0] - p[0]) * fraction,
        p[1] + (q[1] - p[1]) * fraction
    )


def to_acb(point):
    return acb(
        fraction_ball(point[0]),
        fraction_ball(point[1])
    )


def ceil_fraction(x):
    return (
        x.numerator + x.denominator - 1
    ) // x.denominator


# Short panels on the horizontal edges.
# Slightly longer panels on the vertical edges.
panel_lengths = [
    Fraction(1, 10),
    Fraction(1, 2),
    Fraction(1, 10),
    Fraction(1, 2)
]

tol = arb("1e-18")

side_integrals = []

print("SESSION 08: CONTOUR INTEGRATION")
print("--------------------------------")

integration_start = time.time()

for i in range(4):

    if not boundary_certified:
        raise RuntimeError("Boundary not certified.")

    p = corners[i]
    q = corners[i + 1]

    span = max(
        abs(q[0] - p[0]),
        abs(q[1] - p[1])
    )

    n = max(
        1,
        ceil_fraction(span / panel_lengths[i])
    )

    edge_total = acb(0)

    print(f"\nSide {i + 1}: {n} panels")

    for k in range(n):

        p1 = interpolate_point(p, q, k, n)
        p2 = interpolate_point(p, q, k + 1, n)

        result = acb.integral(
            zeta_log_derivative,
            to_acb(p1),
            to_acb(p2),
            abs_tol=tol,
            rel_tol=tol,
            deg_limit=80,
            eval_limit=200000,
            depth_limit=5000,
            use_heap=True
        )

        if not result.is_finite():
            raise RuntimeError(
                f"Integration failed: side {i + 1}, panel {k + 1}"
            )

        edge_total += result

        if (k + 1) % 250 == 0 or k + 1 == n:
            print(f"Completed {k + 1}/{n} panels")

    side_integrals.append(edge_total)

    print("Side integral:", edge_total)


total_integral = sum(side_integrals, acb(0))

print("\nTOTAL CONTOUR INTEGRAL")
print("----------------------")
print(total_integral)

print("\nElapsed seconds:", round(time.time() - integration_start, 2))

SESSION 08: CONTOUR INTEGRATION
--------------------------------

Side 1: 10 panels
Completed 10/10 panels
Side integral: [-0.2323540132923274 +/- 2.00e-17] + [-0.0210732707272523 +/- 1.47e-17]j

Side 2: 1980 panels
Completed 250/1980 panels
Completed 500/1980 panels
Completed 750/1980 panels
Completed 1000/1980 panels
Completed 1250/1980 panels
Completed 1500/1980 panels
Completed 1750/1980 panels
Completed 1980/1980 panels
Side integral: [-0.39234395969933 +/- 7.02e-15] + [0.12683087083839 +/- 4.64e-15]j

Side 3: 10 panels
Completed 10/10 panels
Side integral: [2.5349391062863958 +/- 3.07e-17] + [2.3152907958856373 +/- 5.15e-17]j

Side 4: 1980 panels
Completed 250/1980 panels
Completed 500/1980 panels
Completed 750/1980 panels
Completed 1000/1980 panels
Completed 1250/1980 panels
Completed 1500/1980 panels
Completed 1750/1980 panels
Completed 1980/1980 panels
Side integral: [-1.9102411332947 +/- 7.95e-14] + [4075.3662159635548 +/- 9.33e-14]j

TOTAL CONTOUR INTEGRAL
------------------

In [6]:
#Exact zero count
two_pi_i = 2 * acb.pi() * acb(0, 1)

zero_count_ball = total_integral / two_pi_i

unique_count = zero_count_ball.unique_fmpz()

print("SESSION 08: ZERO COUNT")
print("----------------------")
print("Zero count enclosure:")
print(zero_count_ball)

if unique_count is None:

    print("\nNOT YET CERTIFIED")
    print("The enclosure does not isolate one integer.")

else:

    certified_count = int(unique_count)

    print("\nCERTIFIED ZERO COUNT:", certified_count)
    print("Expected count: 649")
    print("Matches expected:", certified_count == 649)

SESSION 08: ZERO COUNT
----------------------
Zero count enclosure:
[649.00000000000000 +/- 7.65e-15] + [+/- 7.65e-15]j

CERTIFIED ZERO COUNT: 649
Expected count: 649
Matches expected: True


In [7]:
#Generate candidate intervals automatically
print("GENERATING CANDIDATE ZERO INTERVALS")
print("-----------------------------------")

# Generate the first 650 positive-height zero candidates.
seed_zeros = list(acb.zeta_zeros(1, 650))

candidate_records = []

for index, zero in enumerate(seed_zeros, start=1):

    gamma_text = zero.imag.mid().str(
        55,
        radius=False
    )

    gamma = Fraction(gamma_text)

    if T_LO < gamma < T_HI:
        candidate_records.append((index, gamma))


print("Candidates inside region:", len(candidate_records))

assert len(candidate_records) == 649, (
    f"Expected 649 candidates, found {len(candidate_records)}"
)

# Construct narrow, disjoint rational intervals.
radius = Fraction(1, 100)

intervals = [
    (
        index,
        gamma - radius,
        gamma + radius
    )
    for index, gamma in candidate_records
]

assert all(
    intervals[i][2] < intervals[i + 1][1]
    for i in range(len(intervals) - 1)
)

assert all(
    left > T_LO and right < T_HI
    for _, left, right in intervals
)

print("All intervals disjoint: True")
print("All intervals inside (10,1000): True")

print("\nFirst candidate:", float(candidate_records[0][1]))
print("Last candidate:", float(candidate_records[-1][1]))

GENERATING CANDIDATE ZERO INTERVALS
-----------------------------------
Candidates inside region: 649
All intervals disjoint: True
All intervals inside (10,1000): True

First candidate: 14.134725141734695
Last candidate: 999.7915715574129


In [8]:
#Rigorous Hardy Z verification
def hardy_z_ball(t_fraction):

    t = fraction_ball(t_fraction)

    series = arb_series(
        [t, 1],
        prec=2
    ).riemann_siegel_z()

    return series.coeffs()[0]


def certified_sign(value):

    sign = value.sgn()

    if sign == arb(1):
        return "+"

    if sign == arb(-1):
        return "-"

    return "UNKNOWN"


verification_rows = []
verified = 0
failed_intervals = []

print("SESSION 08: HARDY Z VERIFICATION")
print("--------------------------------")

start = time.time()

for i, (zero_index, left, right) in enumerate(intervals, start=1):

    z_left = hardy_z_ball(left)
    z_right = hardy_z_ball(right)

    sign_left = certified_sign(z_left)
    sign_right = certified_sign(z_right)

    passed = (
        sign_left != "UNKNOWN"
        and sign_right != "UNKNOWN"
        and sign_left != sign_right
    )

    if passed:
        verified += 1
    else:
        failed_intervals.append(zero_index)

    verification_rows.append({
        "zero_index": zero_index,
        "left": str(left),
        "right": str(right),
        "left_sign": sign_left,
        "right_sign": sign_right,
        "passed": passed
    })

    if i % 50 == 0 or i == len(intervals):
        print(
            f"Checked {i}/{len(intervals)} | "
            f"Verified: {verified} | "
            f"Failed: {len(failed_intervals)}"
        )


print("\nVERIFIED SIGN CHANGES:", verified)
print("TOTAL INTERVALS:", len(intervals))
print("Failed intervals:", failed_intervals)
print("Elapsed seconds:", round(time.time() - start, 2))

SESSION 08: HARDY Z VERIFICATION
--------------------------------
Checked 50/649 | Verified: 50 | Failed: 0
Checked 100/649 | Verified: 100 | Failed: 0
Checked 150/649 | Verified: 150 | Failed: 0
Checked 200/649 | Verified: 200 | Failed: 0
Checked 250/649 | Verified: 250 | Failed: 0
Checked 300/649 | Verified: 300 | Failed: 0
Checked 350/649 | Verified: 350 | Failed: 0
Checked 400/649 | Verified: 400 | Failed: 0
Checked 450/649 | Verified: 450 | Failed: 0
Checked 500/649 | Verified: 500 | Failed: 0
Checked 550/649 | Verified: 550 | Failed: 0
Checked 600/649 | Verified: 600 | Failed: 0
Checked 649/649 | Verified: 649 | Failed: 0

VERIFIED SIGN CHANGES: 649
TOTAL INTERVALS: 649
Failed intervals: []
Elapsed seconds: 1.47


In [9]:
count_below_10 = arb("10").zeta_nzeros()
count_below_1000 = arb("1000").zeta_nzeros()

independent_count_ball = (
    count_below_1000 - count_below_10
)

independent_count = independent_count_ball.unique_fmpz()

print("INDEPENDENT ZERO-COUNT CHECK")
print("----------------------------")

print("Zeros below 10:", count_below_10)
print("Zeros below 1000:", count_below_1000)
print("Difference:", independent_count_ball)
print("Isolated integer:", independent_count)

INDEPENDENT ZERO-COUNT CHECK
----------------------------
Zeros below 10: 0
Zeros below 1000: 649.000000000000000000000000000000000000000000000000000000
Difference: 649.000000000000000000000000000000000000000000000000000000
Isolated integer: 649


In [10]:
with open(
    "Session08_zero_verification.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "zero_index",
            "left",
            "right",
            "left_sign",
            "right_sign",
            "passed"
        ]
    )

    writer.writeheader()
    writer.writerows(verification_rows)


final_success = (
    boundary_certified
    and unique_count is not None
    and int(unique_count) == 649
    and verified == 649
    and len(failed_intervals) == 0
    and independent_count is not None
    and int(independent_count) == 649
)

report = {
    "session": 8,
    "height_range": [10, 1000],
    "boundary_certified": boundary_certified,
    "contour_zero_count": (
        int(unique_count) if unique_count is not None else None
    ),
    "hardy_sign_changes": verified,
    "independent_zero_count": (
        int(independent_count)
        if independent_count is not None else None
    ),
    "failed_intervals": failed_intervals,
    "success": final_success
}

with open(
    "Session08_verification_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)


print("SESSION 08 FINAL RESULT")
print("=======================")
print(json.dumps(report, indent=2))

if final_success:
    print("\nSESSION 08: SUCCESS")
    print("All 649 zeros in the region are accounted for")
    print("by distinct critical-line zeros.")

else:
    print("\nSESSION 08: NOT YET COMPLETE")
    print("Inspect the failed checks.")

SESSION 08 FINAL RESULT
{
  "session": 8,
  "height_range": [
    10,
    1000
  ],
  "boundary_certified": true,
  "contour_zero_count": 649,
  "hardy_sign_changes": 649,
  "independent_zero_count": 649,
  "failed_intervals": [],
  "success": true
}

SESSION 08: SUCCESS
All 649 zeros in the region are accounted for
by distinct critical-line zeros.
